In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
indicator_name = 'Cost_6'
# estimate = 'ACS1'
estimate = 'ACS5'

In [ ]:
df_p = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_P.csv')
                                , dtype = {'State FIPS': object})
df_h = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H.csv')
                                , dtype = {'State FIPS': object})

# df_p = df_p.drop('RT', axis = 1)
# df_h = df_h.drop('RT', axis = 1)

In [ ]:
unique(df_h[df_h['State FIPS'] == '08']['year'].values)

In [ ]:
df_p.head()

In [ ]:
df_h.head()

In [ ]:

df_p2 = df_p.merge(df_h, on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'], how = 'left')

# df_p2['GRPIP'] = df_p2['GRPIP'].replace(101, np.nan)
# df_p2['OCPIP'] = df_p2['OCPIP'].replace(101, np.nan)

conditions = [
                # ( (df_p2['GRPIP'].isna()) | (df_p2['OCPIP'].isna()) ),
                ( (df_p2['GRPIP'] == 101) | (df_p2['OCPIP'] == 101) ),
                ( ((df_p2['GRPIP'] == 0) & (df_p2['OCPIP'] <= 30)) | ((df_p2['OCPIP'] == 0) & (df_p2['GRPIP'] <= 30)) ),
                ( ((df_p2['GRPIP'] > 30) & (df_p2['GRPIP'] <= 50)) | ((df_p2['OCPIP'] > 30) & (df_p2['OCPIP'] <= 50)) ),
                ( (df_p2['GRPIP']  > 50) | (df_p2['OCPIP']  > 50) )
            ]
choices = ['Cost data not available', 'Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']
df_p2["housing_burden"] = np.select(conditions, choices)

df_p2 = df_p2.drop(['PWGTP', 'GRPIP', 'OCPIP'], axis = 1).drop_duplicates()
df_p2 = df_p2.groupby(['State FIPS', 'PUMA', 'PUMA NAME', 'year', 'RAC1P', 'housing_burden'], as_index = False)['WGTP'].agg(sum)
df_p2.head()

In [ ]:
df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06', '08'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06', '08'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_p2['PUMA'] = df_p2['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA', 'MPO']]

df_p22 = df_p2[df_p2['year'].isin(sequence(2020, 2029))].merge(df_fips2[df_fips2['Years'] == '2020-2029'], on = ['State FIPS', 'PUMA'], how = 'left')
df_p21 = df_p2[df_p2['year'].isin(sequence(2010, 2019))].merge(df_fips2[df_fips2['Years'] == '2010-2019'], on = ['State FIPS', 'PUMA'], how = 'left')
df_p2 = pd.concat([df_p21, df_p22])

df_p2 = df_p2.merge(df_fips1, on = ['State FIPS', 'County FIPS'], how = 'left')

df_p2.head()

In [ ]:
df_p2 = df_p2.groupby(['State FIPS', 'MPO', 'year', 'RAC1P', 'housing_burden'], as_index = False)['WGTP'].agg(sum)
df_p2.head()

In [ ]:
geography = 'PUMA'

# Set output name for .xlsx files
name_output_long_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' Long.xlsx']
name_output_long_xlsx = "".join(name_output_long_xlsx)

# Set output name for .csv files
name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
name_output_PUMA_csv = "".join(name_output_PUMA_csv)

In [ ]:
report_theme = 'Vibrant and Inclusive Places'
sp_folder_out = 'Development\\Housing Cost'

# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
df_p2.to_excel(os.path.join(path_out_xlsx, name_output_long_xlsx), index = False)

print('')
print("Successfully exported")